In [ ]:


# 1. INSTALL REQUIRED LIBRARIES

!pip install -q langchain langchain-groq


# 2. IMPORT LIBRARIES


import os
import csv
from google.colab import userdata

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser


# 3. CONFIGURE GROQ API


api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Add it in Google Colab Secrets."
    )

os.environ["GROQ_API_KEY"] = api_key

MODEL = "openai/gpt-oss-20b"

llm = ChatGroq(
    model=MODEL,
    temperature=0.3
)

print("Groq LLM connected successfully!")


# 4. PRODUCT CATALOGUE


PRODUCTS = {
    "rice": {"price": 60, "unit": "kg"},
    "sugar": {"price": 45, "unit": "kg"},
    "milk": {"price": 30, "unit": "litre"},
    "bread": {"price": 40, "unit": "packet"},
    "eggs": {"price": 7, "unit": "piece"},
    "apples": {"price": 120, "unit": "kg"},
    "tomatoes": {"price": 35, "unit": "kg"},
    "potatoes": {"price": 30, "unit": "kg"},
    "oil": {"price": 150, "unit": "litre"}
}

print("\nAvailable Products:")
for item, details in PRODUCTS.items():
    print(
        f"{item.title()} - Rs.{details['price']} per {details['unit']}"
    )


# 5. SYSTEM PROMPT VERSION 1


SYSTEM_V1 = """
You are a helpful grocery shopping assistant.

Answer customer questions about grocery products.
Be polite and helpful.
Give short and simple answers.
"""


# 6. SYSTEM PROMPT VERSION 2

SYSTEM_V2 = """
You are a Smart Grocery Shopping Assistant.

Your product catalogue is:

Rice - Rs.60 per kg
Sugar - Rs.45 per kg
Milk - Rs.30 per litre
Bread - Rs.40 per packet
Eggs - Rs.7 per piece
Apples - Rs.120 per kg
Tomatoes - Rs.35 per kg
Potatoes - Rs.30 per kg
Oil - Rs.150 per litre

Follow these rules:

1. Give product prices only from the catalogue.
2. Never invent product prices.
3. If a product is not listed, say:
   I will check with the store.
4. Do not claim that a product is currently in stock.
5. Do not promise delivery.
6. Calculate total prices carefully.
7. Remember customer details from the conversation history.
8. Keep answers short and under 50 words.
9. Respond in English or Telugu based on the customer's language.
10. Be polite and helpful.
"""


# 7. CREATE LANGCHAIN PROMPT TEMPLATE


prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_V2),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# Create the LangChain chain

chain = prompt | llm | StrOutputParser()

print("\nLangChain chain created successfully!")


# 8. CONVERSATION MEMORY


history = []

MAX_TURNS = 8

def ask(question):

    global history

    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    # Keep only the latest conversation turns
    history = history[-(MAX_TURNS * 2):]

    return answer


# Test the chatbot

print("\nCHATBOT TEST")

print("Customer: What is the price of rice?")
print("Assistant:", ask("What is the price of rice?"))

print("\nCustomer: What about milk?")
print("Assistant:", ask("What about milk?"))


# 9. ACTIVITY 1 - PROMPT ENGINEERING EXPERIMENT


def run_prompt_test(system_prompt, question):

    test_prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{question}")
    ])

    test_chain = test_prompt | llm | StrOutputParser()

    return test_chain.invoke({
        "question": question
    })


print("\nACTIVITY 1 - PROMPT COMPARISON")

test_question = "What is the price of rice?"

print("\nVersion 1 Response:")
print(run_prompt_test(SYSTEM_V1, test_question))

print("\nVersion 2 Response:")
print(run_prompt_test(SYSTEM_V2, test_question))


# 10. ACTIVITY 2 - MESSAGE CLASSIFICATION


classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a grocery customer message classifier.

Classify the customer message into one of these categories:

PRODUCT_PRICE
PRODUCT_AVAILABILITY
ORDER
COMPLAINT
GENERAL_QUERY

Return only the category name.
"""),
    ("human", "{message}")
])

classifier_chain = classifier_prompt | llm | StrOutputParser()

messages = [
    "What is the price of rice?",
    "Is milk available?",
    "I want to order 2 kg of apples.",
    "The bread I received was damaged.",
    "What are your store timings?"
]

print("\nACTIVITY 2 - MESSAGE CLASSIFICATION")

for message in messages:

    category = classifier_chain.invoke({
        "message": message
    })

    print("\nMessage:", message)
    print("Category:", category.strip())



# 11. ACTIVITY 3 - JSON INFORMATION EXTRACTION


import json

json_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Extract grocery order details from the customer message.

Return only valid JSON with these keys:
item, quantity, unit.

If a value is not mentioned, use null.
Do not include Markdown code fences.
"""),
    ("human", "{message}")
])

json_chain = json_prompt | llm | StrOutputParser()

sample_message = "I want to buy 3 kg of rice."

print("\nACTIVITY 3 - JSON EXTRACTION")

json_response = json_chain.invoke({
    "message": sample_message
})

print("LLM Response:")
print(json_response)

try:

    extracted_data = json.loads(json_response)

    print("\nExtracted Information:")
    print(extracted_data)

except json.JSONDecodeError:

    print("The model did not return valid JSON.")
    print("Raw response:", json_response)



# 12. ACTIVITY 4 - MEMORY EXPERIMENT


print("\nACTIVITY 4 - MEMORY EXPERIMENT")

short_history = []

short_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_V2),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

short_chain = short_prompt | llm | StrOutputParser()


def memory_test(question):

    global short_history

    response = short_chain.invoke({
        "history": short_history,
        "question": question
    })

    short_history.extend([
        ("human", question),
        ("ai", response)
    ])

    # Keep only the latest 2 conversation turns
    short_history = short_history[-4:]

    return response


print("\nCustomer: My name is Anusha.")
print("Assistant:", memory_test("My name is Anusha."))

print("\nCustomer: What is my name?")
print("Assistant:", memory_test("What is my name?"))

print("\nNote: Memory is maintained only during this notebook session.")



# 13. ACTIVITY 5 - CREATE TEST DATASET


test_cases = [
    {
        "question": "What is the price of rice?",
        "must_include": "60"
    },
    {
        "question": "What is the price of milk?",
        "must_include": "30"
    },
    {
        "question": "How much do apples cost?",
        "must_include": "120"
    },
    {
        "question": "What is the price of one egg?",
        "must_include": "7"
    },
    {
        "question": "What is the cost of 3 kg rice?",
        "must_include": "180"
    },
    {
        "question": "What is the cost of 2 litres of milk?",
        "must_include": "60"
    },
    {
        "question": "Do you sell chocolate?",
        "must_include": "check with the store"
    },
    {
        "question": "Is bread currently in stock?",
        "must_include": "check with the store"
    },
    {
        "question": "Can you promise delivery today?",
        "must_include": "check with the store"
    },
    {
        "question": "What is the price of bread?",
        "must_include": "40"
    }
]

# Save test cases into CSV

with open("tests.csv", "w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=["question", "must_include"]
    )

    writer.writeheader()
    writer.writerows(test_cases)

print("\ntests.csv created successfully!")



# 14. RUN AUTOMATED TESTS


def run_tests(system_prompt, temperature=0.3):

    test_llm = ChatGroq(
        model=MODEL,
        temperature=temperature
    )

    test_prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{question}")
    ])

    test_chain = test_prompt | test_llm | StrOutputParser()

    passed = 0

    print("\nRunning tests...")

    for test in test_cases:

        response = test_chain.invoke({
            "question": test["question"]
        })

        expected = test["must_include"]

        success = expected.lower() in response.lower()

        if success:
            passed += 1

        print("\nQuestion:", test["question"])
        print("Expected to include:", expected)
        print("Actual response:", response)
        print("Result:", "PASS" if success else "FAIL")

    total = len(test_cases)

    accuracy = (passed / total) * 100

    print("\nTEST SUMMARY")
    print("Total tests:", total)
    print("Passed:", passed)
    print("Failed:", total - passed)
    print("Pass percentage:", round(accuracy, 2), "%")

    return accuracy



# 15. COMPARE PROMPT V1 AND PROMPT V2


print("\nPROMPT VERSION 1 TEST RESULTS")

accuracy_v1 = run_tests(SYSTEM_V1)

print("\nPROMPT VERSION 2 TEST RESULTS")

accuracy_v2 = run_tests(SYSTEM_V2)



# 16. TEMPERATURE EXPERIMENT


print("\nTEMPERATURE EXPERIMENT")

print("\nTesting with temperature 0.0")

accuracy_low = run_tests(
    SYSTEM_V2,
    temperature=0.0
)

print("\nTesting with temperature 1.0")

accuracy_high = run_tests(
    SYSTEM_V2,
    temperature=1.0
)

print("\nTEMPERATURE COMPARISON")

print("Temperature 0.0:", round(accuracy_low, 2), "%")
print("Temperature 1.0:", round(accuracy_high, 2), "%")



# 17. AGENT CONCEPT PREVIEW


print("\nAGENT CONCEPT PREVIEW")

print("""
An AI agent can:
1. Understand the customer's request.
2. Decide what action is needed.
3. Select an appropriate tool.
4. Execute the tool.
5. Return the result.

Our current LangChain chain sends a prompt to the LLM
and returns its response. It does not yet use grocery tools.
""")

# Optional LangChain agent interface preview

try:

    from langchain.agents import create_agent

    agent = create_agent(
        model=llm,
        tools=[],
        system_prompt=SYSTEM_V2
    )

    agent_response = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": "What is the price of rice?"
            }
        ]
    })

    print("\nAgent interface preview:")
    print(agent_response["messages"][-1].content)

except Exception as error:

    print("\nAgent preview could not run with the installed version.")
    print("The LangChain chain and activities above can still be used.")
    print("Details:", str(error))



# 18. DAY 4 PREPARATION - GROCERY TOOLS


def get_product_price(item: str) -> int:
    """
    Return the sample catalogue price of a grocery product.
    """

    item = item.lower().strip()

    if item not in PRODUCTS:
        raise ValueError("Product not found in catalogue.")

    return PRODUCTS[item]["price"]


def is_product_available(item: str) -> bool:
    """
    Check whether a product is listed in the sample catalogue.
    This does not check real-time store stock.
    """

    item = item.lower().strip()

    return item in PRODUCTS


print("\nDAY 4 PREPARATION")

print("Rice price:", get_product_price("rice"))

print(
    "Is milk listed in catalogue?",
    is_product_available("milk")
)

print(
    "Is chocolate listed in catalogue?",
    is_product_available("chocolate")
)



# 19. FINAL CHATBOT INTERACTION


print("\n" + "=" * 50)
print("SMART GROCERY SHOPPING ASSISTANT")
print("=" * 50)

print("Ask me about grocery products and prices.")
print("Type 'exit' to stop the chatbot.")

while True:

    user_input = input("\nYou: ").strip()

    if user_input.lower() == "exit":
        print("Assistant: Thank you for using our grocery assistant!")
        break

    if not user_input:
        print("Assistant: Please enter a message.")
        continue

    try:

        response = ask(user_input)

        print("Assistant:", response)

    except Exception as error:

        print("Something went wrong:", str(error))


print("\nDAY 3 TASK EXECUTION COMPLETED!")


Groq LLM connected successfully!

Available Products:
Rice - Rs.60 per kg
Sugar - Rs.45 per kg
Milk - Rs.30 per litre
Bread - Rs.40 per packet
Eggs - Rs.7 per piece
Apples - Rs.120 per kg
Tomatoes - Rs.35 per kg
Potatoes - Rs.30 per kg
Oil - Rs.150 per litre

LangChain chain created successfully!

CHATBOT TEST
Customer: What is the price of rice?
Assistant: Rice costs Rs.60 per kg.

Customer: What about milk?
Assistant: Milk costs Rs.30 per litre.

ACTIVITY 1 - PROMPT COMPARISON

Version 1 Response:
The price of rice varies by type, brand, and size.  
- White long‑grain: about $1–$3 per pound  
- Brown rice: $1.50–$4 per pound  
- Specialty or organic: $3–$6 per pound  

If you have a specific brand or package size in mind, let me know and I can give a more precise estimate.

Version 2 Response:
Rice is Rs.60 per kg.

ACTIVITY 2 - MESSAGE CLASSIFICATION

Message: What is the price of rice?
Category: PRODUCT_PRICE

Message: Is milk available?
Category: PRODUCT_AVAILABILITY

Message: I w